In [ ]:
import json
import csv
import os
# Load multiple JSON objects from a file
multi_run_dir = "output/multi_run/"

for filename in os.listdir(multi_run_dir):
    if filename.endswith(".json"):
        file_path = os.path.join(multi_run_dir, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            # Process your data here
            print(f"Processed {filename}: {type(data)}")

        # Determine max number of quotes across all entries
        max_quotes = 0
        for entry in data:
            if "answer_facts" in entry and entry["answer_facts"]:
                max_quotes = max(max_quotes, len(entry["answer_facts"][0].get("substring_quote", [])))

        # Prepare output CSV filename
        output_csv = f"{os.path.splitext(filename)[0]}_output.csv"
        print(output_csv)
        output_csv_path = os.path.join(multi_run_dir, output_csv)
        print(output_csv_path)
        
        # Prepare CSV
        with open(output_csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            # Header
            header = ["entity", "exist", "value", "fact"]
            for i in range(1, max_quotes + 1):
                header += [f"quote{i}", f"index{i}"]
            writer.writerow(header)

            for entry in data:
                for key in entry:
                    if key.endswith("_exist"):
                        entity = key.replace("_exist", "")
                        exist = entry[key]
                        value = entry.get(entity, "")

                        # Check and extract facts
                        if "answer_facts" in entry and entry["answer_facts"]:
                            fact_obj = entry["answer_facts"][0]
                            fact = fact_obj.get("fact", "")
                            quotes = fact_obj.get("substring_quote", [])
                            indices = fact_obj.get("substring_indices", [])
                        else:
                            fact = ""
                            quotes = []
                            indices = []

                        row = [entity, exist, value, fact]
                        for q, idx in zip(quotes, indices):
                            row.extend([q, str(idx)])  # you can use `json.dumps(idx)` for JSON format

                        # Pad to match header length
                        while len(row) < len(header):
                            row.append("")
                        writer.writerow(row)


Processed NCT04915612_EC_run_1.json: <class 'list'>
NCT04915612_EC_run_1_output.csv
output/multi_run/NCT04915612_EC_run_1_output.csv
Processed NCT04898894_IC_run_1.json: <class 'list'>
NCT04898894_IC_run_1_output.csv
output/multi_run/NCT04898894_IC_run_1_output.csv
Processed NCT04915612_IC_run_1.json: <class 'list'>
NCT04915612_IC_run_1_output.csv
output/multi_run/NCT04915612_IC_run_1_output.csv
Processed NCT04898894_EC_run_1.json: <class 'list'>
NCT04898894_EC_run_1_output.csv
output/multi_run/NCT04898894_EC_run_1_output.csv
Processed NCT05188170_EC_run_1.json: <class 'list'>
NCT05188170_EC_run_1_output.csv
output/multi_run/NCT05188170_EC_run_1_output.csv
Processed NCT05188170_IC_run_1.json: <class 'list'>
NCT05188170_IC_run_1_output.csv
output/multi_run/NCT05188170_IC_run_1_output.csv


In [7]:
import json
import pandas as pd

# === Load files ===
with open("output/multi_run/NCT04915612_IC_run_1_20250824_151413.json", "r") as f_ic, \
     open("output/multi_run/NCT04915612_EC_run_1_20250824_151413.json", "r") as f_ec:
    ic_data = json.load(f_ic)
    ec_data = json.load(f_ec)

# === Convert list to ordered variable dict ===
def extract_ordered_dict(data, suffix):
    ordered_dict = {}
    for item in data:
        for key in item:
            if key.endswith("_exist"):
                var_name = key.replace("_exist", "")
                ordered_dict[var_name] = {
                    f"Exist_{suffix}": item.get(f"{var_name}_exist"),
                    f"Value_{suffix}": item.get(var_name),
                    f"Reasoning_{suffix}": item.get("reasoning")
                }
                break
    return ordered_dict

# === Extract IC and EC while preserving order ===
ic_dict = extract_ordered_dict(ic_data, "IC")
ec_dict = extract_ordered_dict(ec_data, "EC")

# === Preserve order: IC first, then EC-only variables ===
all_vars = list(ic_dict.keys()) + [v for v in ec_dict.keys() if v not in ic_dict]

# === Merge rows ===
rows = []
for var in all_vars:
    row = {
        "Variable": var,
        "Exist_IC": ic_dict.get(var, {}).get("Exist_IC"),
        "Value_IC": ic_dict.get(var, {}).get("Value_IC"),
        "Exist_EC": ec_dict.get(var, {}).get("Exist_EC"),
        "Value_EC": ec_dict.get(var, {}).get("Value_EC"),
        "Reasoning_IC": ic_dict.get(var, {}).get("Reasoning_IC"),
        "Reasoning_EC": ec_dict.get(var, {}).get("Reasoning_EC")
    }
    rows.append(row)

# === Create DataFrame ===
final_df = pd.DataFrame(rows).set_index("Variable")
final_df.to_csv("output/multi_run/test.csv")
# === Display or Save ===
print(final_df.head())  # Print first few rows
# final_df.to_csv("merged_output.csv")  # Optional save

           Exist_IC        Value_IC Exist_EC  Value_EC  \
Variable                                                 
age_min         Yes               0       No      None   
age_max         Yes              21       No      None   
weight_min       No            None       No      None   
weight_max       No            None       No      None   
sex             Yes  [Male, Female]      Yes  [Female]   

                                                 Reasoning_IC Reasoning_EC  
Variable                                                                    
age_min     The inclusion criteria explicitly state 'Age =...         None  
age_max     The inclusion criteria explicitly state 'Age =...         None  
weight_min  The inclusion criteria do not mention any mini...         None  
weight_max  The inclusion criteria do not mention any maxi...         None  
sex         The inclusion criteria explicitly mention requ...         None  


In [5]:
final_df

,Exist_IC,Value_IC,Exist_EC,Value_EC,Reasoning
Variable,,,,,
age_min,Yes,0,No,None,The inclusion criteria explicitly state 'Age =...
age_max,Yes,21,No,None,The inclusion criteria explicitly state 'Age =...
weight_min,No,None,No,None,The inclusion criteria do not mention any mini...
weight_max,No,None,No,None,The inclusion criteria do not mention any maxi...
sex,Yes,"[Male, Female]",Yes,[Female],The inclusion criteria explicitly mention requ...
...,...,...,...,...,...
expression_alk,No,None,No,None,The inclusion criteria do not mention ALK expr...
expression_alk_type,No,None,No,None,The inclusion criteria do not mention ALK rear...
bclc_stage,No,None,No,None,The inclusion criteria do not mention BCLC sta...
